# Recommendation System for web articles in Mondadori Group
---

In [22]:
# importing
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

In [23]:
# setting
model_name = "nickprock/sentence-bert-base-italian-xxl-uncased"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [24]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(32102, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [ ]:
def mean_pooling(model_output, attention_mask):
    """
    Apply mean pooling on token embeddings, considering the attention mask
    """
    token_embeddings = model_output[0] # all token embeddings
    # expand attenion mask to multiply the embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    # weighted sum and normalize to obtain averaged sentence embedding
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask

# Definisci alcune frasi di esempio
texts = [
    "Una ragazza si acconcia i capelli.",
    "Una ragazza si sta spazzolando i capelli.",
    "La prossima settimana accorcerò un po' i capelli: sono diventati troppo lunghi.",
    "Mi piacciono i libri gialli.",
    "Sono appassionato di UFO.",
    "La signora in giallo è la mia serie preferita."
]

# Tokenizza le frasi con padding e troncamento
encoded_input = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
# Sposta tutti i tensori sul device scelto
encoded_input = {key: tensor.to(device) for key, tensor in encoded_input.items()}

# Esegui il modello in modalità inference
with torch.no_grad():
    model_output = model(**encoded_input)

# Applica il mean pooling per ottenere un embedding per ciascuna frase
sentence_embeddings = mean_pooling(model_output, encoded_input["attention_mask"])

In [37]:
# Frase di input arbitraria
input_sentence = "ET è il mio film preferito."

# Tokenizza e calcola l'embedding per la frase di input
encoded_input_sentence = tokenizer(input_sentence, padding=True, truncation=True, return_tensors="pt")
# Sposta i tensori sullo stesso device usato precedentemente (ad es. "mps" o "cpu")
encoded_input_sentence = {key: tensor.to(device) for key, tensor in encoded_input_sentence.items()}


In [38]:
# Esegui il modello in modalità inference
with torch.no_grad():
    model_output = model(**encoded_input_sentence)

In [39]:
input_embedding = mean_pooling(model_output, encoded_input_sentence["attention_mask"])
cosine_similarities = F.cosine_similarity(input_embedding, sentence_embeddings, dim=1)

In [40]:
cosine_similarities

tensor([0.0381, 0.0205, 0.0947, 0.3286, 0.2103, 0.3407], device='mps:0')